# VLM-Anomaly — Full MVTec Sweep · PaDiM (Classical Baseline)

**Model:** `padim` via [Anomalib](https://github.com/openvinotoolkit/anomalib) (Intel, Apache 2.0)  
**Strategy:** Train on normal split → evaluate on test split. No GPU needed.  
**Cost:** **$0** — fully open-source, runs on CPU.  

| Scope | Est. time (CPU) | Cost |
|---|---|---|
| 1 category | ~1 min | $0 |
| Full sweep (15 cat) | ~15 min | $0 |


In [1]:
# ── Cell 1: Setup paths & sys.path ─────────────────────────────────────────
import sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT   = Path().resolve().parent
SRC_DIR     = REPO_ROOT / 'src'
RESULTS_DIR = REPO_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert SRC_DIR.exists(), f'src/ not found at {SRC_DIR}'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

load_dotenv(REPO_ROOT / '.env')

import vlm_anomaly
print(f'vlm_anomaly {vlm_anomaly.__version__} ready')
print(f'Results : {RESULTS_DIR}')


vlm_anomaly 0.1.0 ready
Results : /Users/sabareeswarans/Projects_26/VLM-Anomaly/results


In [2]:
# ── Cell 2: Verify anomalib + torch ─────────────────────────────────────────
try:
    import anomalib, torch
    print(f'anomalib {anomalib.__version__} | torch {torch.__version__}')
except ImportError as e:
    raise ImportError(
        f'Missing dependency: {e}\n'
        'Install with: pip install anomalib torch==2.2.2 torchvision timm\n'
        'Or: uv pip install -e \".[classical]\"'
    )


anomalib 2.4.2 | torch 2.2.2


In [3]:
# ── Cell 3: Find MVTec dataset ──────────────────────────────────────────────
MVTEC_ROOT = None
for candidate in [
    REPO_ROOT / 'data' / 'mvtec',
    REPO_ROOT / 'data' / 'mvtec-ad',
    Path('/tmp/mvtec'),
]:
    if candidate.exists() and any(candidate.iterdir()):
        MVTEC_ROOT = candidate
        break

assert MVTEC_ROOT, f'MVTec not found. Expected at {REPO_ROOT}/data/mvtec'
categories = sorted([d.name for d in MVTEC_ROOT.iterdir() if d.is_dir()])
print(f'MVTec root : {MVTEC_ROOT}')
print(f'Categories : {categories}')


MVTec root : /Users/sabareeswarans/Projects_26/VLM-Anomaly/data/mvtec
Categories : ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']


In [4]:
# ── Cell 4: Configure ───────────────────────────────────────────────────────
MODEL_NAME  = 'padim'
MODEL_ID    = f'classical/{MODEL_NAME}'
IMAGE_SIZE  = 256
DATASET     = 'mvtec'

print(f'Model    : {MODEL_NAME}  ({MODEL_ID})')
print(f'Img size : {IMAGE_SIZE}x{IMAGE_SIZE}')
print(f'Cost     : $0 (open-source, CPU only)')


Model    : padim  (classical/padim)
Img size : 256x256
Cost     : $0 (open-source, CPU only)


In [5]:
# ── Cell 5: Build shared objects ────────────────────────────────────────────
from vlm_anomaly.config import Settings
from vlm_anomaly.logging import configure_logging
from vlm_anomaly.evaluators.classical_evaluator import ClassicalEvaluator

configure_logging(log_level='INFO')

settings = Settings(
    _env_file=str(REPO_ROOT / '.env'),
    data_dir=str(MVTEC_ROOT.parent),
    results_dir=str(RESULTS_DIR),
)

print(f'Settings ready — data_dir={settings.data_dir}')


Settings ready — data_dir=/Users/sabareeswarans/Projects_26/VLM-Anomaly/data


In [6]:
# ── Cell 5b: Smoke test — 1 category before full sweep ──────────────────────
# Optional — run this to verify the setup before the full 15-category sweep.
import time, json as _json

smoke_cat = 'bottle'
print(f'Smoke test: {MODEL_NAME} / {smoke_cat} / CPU ...')
t0 = time.perf_counter()
ev = ClassicalEvaluator(
    model_name=MODEL_NAME, dataset_name=DATASET,
    category=smoke_cat, image_size=IMAGE_SIZE, settings=settings,
)
smoke_result = ev.run()
print(f'  AUROC    : {smoke_result.auroc:.4f}')
print(f'  F1       : {smoke_result.f1:.4f}')
print(f'  elapsed  : {(time.perf_counter()-t0):.0f}s')
print(f'  cost     : $0')
print('PASS' if smoke_result.auroc and smoke_result.auroc > 0.5 else 'FAIL — check setup')


Smoke test: padim / bottle / CPU ...


2026-05-24T05:20:03.912862Z [info     ] Initializing Padim model.      [anomalib.models.components.base.anomalib_module]
2026-05-24T05:20:04.243733Z [info     ] Loading pretrained weights from Hugging Face hub (timm/resnet18.a1_in1k) [timm.models._builder]
2026-05-24T05:20:04.393789Z [info     ] HTTP Request: HEAD https://huggingface.co/timm/resnet18.a1_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found" [httpx]
2026-05-24T05:20:04.396811Z [info     ] [timm/resnet18.a1_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors. [timm.models._hub]
2026-05-24T05:20:04.485948Z [info     ] Missing keys (fc.weight, fc.bias) discovered while loading pretrained weights. This is expected if model is being adapted. [timm.models._builder]
2026-05-24T05:20:04.521803Z [info     ] classical.run.start            [vlm_anomaly.evaluators.classical_evaluator] category=bottle dataset=mvtec image_size=256 model=padim
2026-05-24T05:20:04.61169

┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor  │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator     │      0 │ train │     0 │
│ 3 │ model          │ PadimModel    │  2.8 M │ train │     0 │
└───┴────────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 2.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.8 M                                                                                                
Total estimated model params size (MB): 11.131                                                                     
Modules in train mode: 19                                                                                          
Modules in eval mode: 69                                                                                           
Total FLOPs: 0

Output()

...

2026-05-24T05:20:39.505405Z [info     ] The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor [lightning_fabric.utilities.rank_zero]
2026-05-24T05:20:39.506870Z [info     ] Found the dataset.             [anomalib.data.datamodules.image.mvtecad]


Output()

2026-05-24T05:20:55.576186Z [info     ] Testing took 16.062868118286133 seconds
Throughput (batch_size=32) : 5.167196753954044 FPS [anomalib.callbacks.timer]


2026-05-24T05:20:55.608705Z [info     ] classical.result.written       [vlm_anomaly.evaluators.classical_evaluator] path=/Users/sabareeswarans/Projects_26/VLM-Anomaly/results/08bf3419_mvtec_bottle_padim.json
2026-05-24T05:20:55.609686Z [info     ] classical.run.done             [vlm_anomaly.evaluators.classical_evaluator] auroc=0.9984127283096313 category=bottle elapsed_s=51.1 f1=0.9841269850730896 model=padim


  AUROC    : 0.9984
  F1       : 0.9841
  elapsed  : 62s
  cost     : $0
PASS


In [7]:
# ── Cell 6: Run all 15 categories (idempotent, resumable) ───────────────────
import json as _json
from tqdm.auto import tqdm

all_results = []


def _already_done(results_dir, category, model_id, dataset):
    """Return path if a complete result file for this model+category exists."""
    for f in results_dir.glob(f'*_{dataset}_{category}_*.json'):
        try:
            data = _json.loads(f.read_text())
            rows = data if isinstance(data, list) else [data]
            if any(r.get('model_id') == model_id for r in rows):
                return f
        except Exception:
            pass
    return None


for category in tqdm(categories, desc=f'{MODEL_NAME} sweep'):
    done = _already_done(RESULTS_DIR, category, MODEL_ID, DATASET)
    if done:
        print(f'  [skip] {category} — already done ({done.name})')
        data = _json.loads(done.read_text())
        all_results.extend(data if isinstance(data, list) else [data])
        continue

    ev = ClassicalEvaluator(
        model_name=MODEL_NAME, dataset_name=DATASET,
        category=category, image_size=IMAGE_SIZE, settings=settings,
    )
    result = ev.run()
    all_results.append(result.model_dump())
    print(
        f'  {category:12s}  AUROC={result.auroc:.3f}  F1={result.f1:.3f}  '
        f'elapsed={result.mean_latency_ms/1000:.0f}s'
    )

print(f'\nDone — {len(all_results)} categories processed.')


padim sweep:   0%|          | 0/15 [00:00<?, ?it/s]

2026-05-24T05:20:55.701861Z [info     ] Initializing Padim model.      [anomalib.models.components.base.anomalib_module]


  [skip] bottle — already done (9f2d4903_mvtec_bottle_padim.json)


2026-05-24T05:20:56.024589Z [info     ] Loading pretrained weights from Hugging Face hub (timm/resnet18.a1_in1k) [timm.models._builder]
2026-05-24T05:20:56.109535Z [info     ] HTTP Request: HEAD https://huggingface.co/timm/resnet18.a1_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found" [httpx]
2026-05-24T05:20:56.112927Z [info     ] [timm/resnet18.a1_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors. [timm.models._hub]
2026-05-24T05:20:56.173824Z [info     ] Missing keys (fc.weight, fc.bias) discovered while loading pretrained weights. This is expected if model is being adapted. [timm.models._builder]
2026-05-24T05:20:56.182912Z [info     ] classical.run.start            [vlm_anomaly.evaluators.classical_evaluator] category=cable dataset=mvtec image_size=256 model=padim
2026-05-24T05:20:56.234228Z [info     ] GPU available: False, used: False [lightning_fabric.utilities.rank_zero]
2026-05-24T05:20:56.235178Z [info 

┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor  │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator     │      0 │ train │     0 │
│ 3 │ model          │ PadimModel    │  2.8 M │ train │     0 │
└───┴────────────────┴───────────────┴────────┴───────┴───────┘

...

2026-05-24T05:39:23.286592Z [info     ] The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor [lightning_fabric.utilities.rank_zero]
2026-05-24T05:39:23.287904Z [info     ] Found the dataset.             [anomalib.data.datamodules.image.mvtecad]


Output()

2026-05-24T05:39:53.243046Z [info     ] Testing took 29.94832992553711 seconds
Throughput (batch_size=32) : 5.042017380449701 FPS [anomalib.callbacks.timer]


2026-05-24T05:39:53.294499Z [info     ] classical.result.written       [vlm_anomaly.evaluators.classical_evaluator] path=/Users/sabareeswarans/Projects_26/VLM-Anomaly/results/edd61325_mvtec_zipper_padim.json
2026-05-24T05:39:53.295540Z [info     ] classical.run.done             [vlm_anomaly.evaluators.classical_evaluator] auroc=0.8841912150382996 category=zipper elapsed_s=73.6 f1=0.942148745059967 model=padim


  zipper        AUROC=0.884  F1=0.942  elapsed=74s

Done — 15 categories processed.


In [8]:
# ── Cell 7: Summary table ────────────────────────────────────────────────────
import pandas as pd

if all_results:
    df = pd.DataFrame(all_results)
    num_cols = ['auroc','f1','precision','recall','mean_latency_ms']
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    print(f'Model    : {MODEL_ID}')
    print(f'Dataset  : {DATASET}')
    print(f'Mean AUROC : {df.auroc.mean():.4f}')
    print(f'Mean F1    : {df.f1.mean():.4f}')
    print(f'Avg time/cat : {df.mean_latency_ms.mean()/1000:.0f}s')
    print(f'Total cost : $0.00  (open-source)')
    print()
    display(df[['category','auroc','f1','mean_latency_ms']].sort_values('auroc', ascending=False).reset_index(drop=True))
else:
    print('No results yet — run Cell 6 first.')


Model    : classical/padim
Dataset  : mvtec
Mean AUROC : 0.9096
Mean F1    : 0.9221
Avg time/cat : 72s
Total cost : $0.00  (open-source)



,category,auroc,f1,mean_latency_ms
0,leather,0.997622,0.983607,79811.035832
1,bottle,0.996032,0.976378,54337.832722
2,carpet,0.991172,0.971098,87290.055511
3,metal_nut,0.976051,0.961749,56939.223337
4,transistor,0.968750,0.904762,68474.603519
5,wood,0.958772,0.935484,63588.735111
6,tile,0.918110,0.915663,64989.972921
7,pill,0.889798,0.946619,81243.201852
8,grid,0.886383,0.900000,64660.932721
9,capsule,0.885521,0.946903,80306.614716


In [9]:
# ── Cell 8: Generate / update report ─────────────────────────────────────────
import importlib, vlm_anomaly.analysis.report_generator as _rg_mod
importlib.reload(_rg_mod)
from vlm_anomaly.analysis.report_generator import generate

REPORT = REPO_ROOT / 'REPORT.md'
generate(RESULTS_DIR, REPORT)
print(f'Report written → {REPORT}')


2026-05-24T05:39:54.312031Z [info     ] report.generate.start          [vlm_anomaly.analysis.report_generator] results_dir=/Users/sabareeswarans/Projects_26/VLM-Anomaly/results
2026-05-24T05:39:59.470082Z [info     ] report.generate.done           [vlm_anomaly.analysis.report_generator] plots=/Users/sabareeswarans/Projects_26/VLM-Anomaly/results/plots report=/Users/sabareeswarans/Projects_26/VLM-Anomaly/REPORT.md


Report written → /Users/sabareeswarans/Projects_26/VLM-Anomaly/REPORT.md


In [10]:
# # ── Cell 9: Show result files + commit hint ──────────────────────────────────
# result_files = sorted(RESULTS_DIR.glob(f'*_{DATASET}_*_{MODEL_NAME}.json'))
# print(f'Result files ({len(result_files)}):')
# for f in result_files:
#     print(f'  {f.name}  ({f.stat().st_size/1024:.1f} KB)')

# print()
# print('To commit:')
# print(f'  git add results/*_{MODEL_NAME}.json')
# print(f'  git commit -m "results({MODEL_ID}): MVTec sweep"')
